# Real-fluid meanline design of a supersonic axial ORC turbine stage

**KCORC Summer School 2026 — guided student notebook**

This notebook is deliberately designed so that **previous Python experience is not required**.

You will not write functions, loops, classes, or property routines. The numerical model is already provided. In only a few orange cells, your task is to translate a turbomachinery equation into a short Python expression.

<div style="padding:0.9rem 1rem;background:#eef4ff;border-left:6px solid #1565c0;border-radius:5px">
<b>Rule for the hands-on part:</b><br>
Only edit lines marked <code>EDIT HERE</code>. Everything else can simply be run with <b>Shift + Enter</b>.
If your answer is not ready, continue anyway — the following sections use the reference implementation, so nobody gets blocked.
</div>

### The four equation-to-Python tasks

1. Cycle boundary conditions $\rightarrow$ pressure ratio and $U/c_{\mathrm{is}}$
2. Isentropic expansion $\rightarrow$ velocity and Mach number
3. Stator continuity $\rightarrow$ throat and outlet area
4. Velocity triangle and Euler work

The final off-design section is **run-only**. It produces the reduced turbine characteristic used in the following TESPy workshop.

## A tiny Python cheat sheet

You only need these translations:

| Mathematics | Python |
|---|---|
| $ab$ | `a * b` |
| $a/b$ | `a / b` |
| $a^2$ | `a**2` |
| $\sqrt{x}$ | `np.sqrt(x)` |
| $\sin(\alpha)$ | `np.sin(alpha)` |
| degrees $\rightarrow$ radians | `np.radians(alpha_deg)` |
| $\tan^{-1}(y/x)$ with correct quadrant | `np.arctan2(y, x)` |

Variable names already include the unit where useful, for example `p_in_Pa`, `U_m_s`, or `dh_is_J_kg`.

## 0. Environment and imports

The Summer School repository is started with the common `uv`/JupyterLab setup.  
Run the next cell once. You do **not** need to understand the imports.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kcorc_turbine import (
    CoolPropBackend,
    DesignInputs,
    design_stage,
    evaluate_rotor_offdesign,
    evaluate_stator_offdesign,
    freeze_geometry,
    inlet_temperature_for_pressure,
)
from kcorc_turbine.checks import checkpoint, compare_with_reference
from kcorc_turbine.plotting import (
    plot_loss_breakdown,
    plot_nozzle_expansion,
    plot_nicfd_diagnostics,
    plot_velocity_triangles,
)
from kcorc_turbine.widgets import design_explorer

backend = CoolPropBackend()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing pyproject.toml"
    )

REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")

## 1. Start with the complete machine

Before deriving anything, use the interactive design explorer.

Change **one slider at a time** and first predict what should happen. Useful questions:

- What happens when rotational speed changes?
- What happens when the rotor inlet metal angle no longer matches the incoming flow angle?
- How does pressure ratio affect nozzle Mach number?
- Can you reduce rotor exit swirl by changing $\beta_2$?

No coding is required here.

In [ ]:
default_inputs = DesignInputs(
    fluid="R1233zd(E)",
    p_in_Pa=2000e3,
    T_in_K=150.0 + 273.15,
    p_out_Pa=500e3,
    m_dot_kg_s=5.025,
    n_rpm=10000.0,
    D_mid_m=0.2,
    alpha_stator_deg=11.5,
    beta_rotor_in_deg=27.0,
    beta_rotor_out_deg=27.0,
    partial_admission=1.0,
    rotor_solidity=1.0,
    rotor_chord_m=0.025,
)

design_explorer(default_inputs)

---
# Block A — From cycle conditions to stage loading

The cycle provides the turbine inlet state, outlet pressure, and mass flow.

For the ideal reference expansion,

$$
s_{\mathrm{out,is}} = s_{\mathrm{in}},
$$

$$
\Delta h_{\mathrm{is}}
=
h_{\mathrm{in}}-h_{\mathrm{out,is}},
$$

and the corresponding velocity scale is

$$
c_{\mathrm{is}}
=
\sqrt{2\Delta h_{\mathrm{is}}}.
$$

The rotor blade speed at the mean diameter is

$$
U = \frac{\pi D_{\mathrm{mid}} n}{60}.
$$

A useful stage-loading indicator is

$$
\frac{U}{c_{\mathrm{is}}}.
$$

In [ ]:
inputs = default_inputs

# The robust reference model is evaluated once and will keep the notebook moving
# even if one of the small student exercises is unfinished.
reference_design = design_stage(inputs, backend=backend)

state_in = reference_design.states["inlet"]
state_out_is = reference_design.states["outlet_isentropic"]

pd.Series({
    "p_in_bar": inputs.p_in_Pa / 1e5,
    "T_in_degC": inputs.T_in_K - 273.15,
    "p_out_bar": inputs.p_out_Pa / 1e5,
    "m_dot_kg_s": inputs.m_dot_kg_s,
    "n_rpm": inputs.n_rpm,
    "D_mid_m": inputs.D_mid_m,
})

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>EQUATION → PYTHON 1</b><br>
Translate the five equations above. Only replace <code>np.nan</code> on the lines marked <code>EDIT HERE</code>.
</div>

In [ ]:
# EDIT HERE
student_pressure_ratio = np.nan

# EDIT HERE
student_dh_is_J_kg = np.nan

# EDIT HERE
student_c_is_m_s = np.nan

# EDIT HERE
student_U_m_s = np.nan

# EDIT HERE
student_U_over_cis = np.nan

pd.Series({
    "pressure_ratio": student_pressure_ratio,
    "dh_is_kJ_kg": student_dh_is_J_kg / 1e3,
    "c_is_m_s": student_c_is_m_s,
    "U_m_s": student_U_m_s,
    "U_over_cis": student_U_over_cis,
})

In [ ]:
compare_with_reference(
    "Pressure ratio",
    student_pressure_ratio,
    reference_design.performance["pressure_ratio"],
)
compare_with_reference(
    "Isentropic velocity",
    student_c_is_m_s,
    reference_design.performance["c_is_m_s"],
    unit="m/s",
)
compare_with_reference(
    "Blade-speed ratio U/c_is",
    student_U_over_cis,
    reference_design.performance["U_over_cis"],
)

> **STOP — joint discussion**
>
> At fixed diameter and cycle conditions, what changes if the shaft speed doubles?  
> Does pressure ratio change just because the shaft speed changes?  
> Why is $U/c_{\mathrm{is}}$ more informative than rpm alone?

---
# Block B — Supersonic stator and sonic throat

Along an ideal adiabatic nozzle, the static state follows the inlet entropy.

At any point along that isentrope,

$$
c_{\mathrm{is}}(p)
=
\sqrt{2\left[h_{\mathrm{in}}-h(p,s_{\mathrm{in}})\right]},
$$

and

$$
Ma(p)=\frac{c_{\mathrm{is}}(p)}{a(p,s_{\mathrm{in}})}.
$$

The sonic throat is reached when $Ma=1$.

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>EQUATION → PYTHON 2</b><br>
Do the calculation only for the ideal nozzle outlet. The complete pressure path is generated for you afterwards.
</div>

In [ ]:
h_in_J_kg = state_in["h_J_kg"]
h_out_is_J_kg = state_out_is["h_J_kg"]
a_out_is_m_s = state_out_is["a_m_s"]

# EDIT HERE
student_c_exit_is_m_s = np.nan

# EDIT HERE
student_Ma_exit_is = np.nan

pd.Series({
    "c_exit_is_m_s": student_c_exit_is_m_s,
    "Ma_exit_is": student_Ma_exit_is,
})

In [ ]:
compare_with_reference(
    "Ideal nozzle exit velocity",
    student_c_exit_is_m_s,
    reference_design.performance["c_is_m_s"],
    unit="m/s",
)
compare_with_reference(
    "Ideal nozzle exit Mach",
    student_Ma_exit_is,
    reference_design.performance["Ma_out_is"],
)

## Follow the complete real-fluid isentrope

The property calls and pressure sweep are already implemented in the reference model.  
We now use the path only for physical interpretation.

In [ ]:
nozzle_path = reference_design.path.copy()

sonic_indices = np.flatnonzero(nozzle_path["Ma_is"].to_numpy() >= 1.0)

if len(sonic_indices):
    throat_index = int(sonic_indices[0])
    is_choked = True
else:
    throat_index = int(np.argmax(nozzle_path["Ma_is"].to_numpy()))
    is_choked = False

throat = nozzle_path.iloc[throat_index]

print(f"Choked: {is_choked}")
print(f"Sonic-throat pressure: {throat['p_Pa']/1e3:.1f} kPa")
print(f"Ideal exit Mach: {nozzle_path.iloc[-1]['Ma_is']:.3f}")

fig, _ = plot_nozzle_expansion(nozzle_path, throat_index)
plt.show()

## A real-fluid surprise: the speed of sound can increase during expansion

For an isentrope, the fundamental derivative of gas dynamics is

$$
\Gamma
=
1+
\frac{\rho}{a}
\left(\frac{\partial a}{\partial \rho}\right)_s.
$$

During expansion $d\rho<0$:

- $\Gamma>1$: the speed of sound decreases in the classical way,
- $0<\Gamma<1$: the speed of sound can increase during expansion,
- $\Gamma<0$: genuinely non-classical/BZT-type behaviour is possible.

This is one reason why a dense-vapour ORC nozzle should not simply be treated as an ideal-gas nozzle.

In [ ]:
nicfd_summary = pd.Series({
    "a_in_m_s": nozzle_path.iloc[0]["a_is_m_s"],
    "a_throat_m_s": nozzle_path.iloc[throat_index]["a_is_m_s"],
    "a_out_m_s": nozzle_path.iloc[-1]["a_is_m_s"],
    "Gamma_min": nozzle_path["Gamma"].min(),
    "Gamma_at_throat": nozzle_path.iloc[throat_index]["Gamma"],
    "Gamma_out": nozzle_path.iloc[-1]["Gamma"],
})
display(nicfd_summary)

for fig, _ in plot_nicfd_diagnostics(nozzle_path):
    plt.show()
    plt.close(fig)

## Stator losses

The detailed empirical loss correlation is supplied as part of the teaching backend.  
You do not need to reproduce its polynomial coefficients.

The velocity coefficient $\phi_s$ converts the ideal enthalpy drop into the loss-adjusted nozzle velocity.

In [ ]:
pd.Series({
    "phi_stator": reference_design.performance["phi_stator"],
    "Ma_out_is": reference_design.performance["Ma_out_is"],
    "Ma_out_loss_adjusted": reference_design.performance["Ma_nozzle_out"],
    "rho_nozzle_out_kg_m3": reference_design.states["nozzle_outlet"]["rho_kg_m3"],
})

## Continuity and nozzle area

For a plane normal to the local velocity,

$$
\dot m = \rho c A,
$$

therefore

$$
A=\frac{\dot m}{\rho c}.
$$

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>EQUATION → PYTHON 3</b><br>
Use continuity once at the sonic throat and once at the loss-adjusted stator outlet.
</div>

In [ ]:
rho_star_kg_m3 = throat["rho_is_kg_m3"]
c_star_m_s = throat["c_is_m_s"]

rho_nozzle_out_kg_m3 = reference_design.states["nozzle_outlet"]["rho_kg_m3"]
c_nozzle_out_m_s = reference_design.velocities["c1_m_s"]

# EDIT HERE
student_A_throat_m2 = np.nan

# EDIT HERE
student_A_outlet_m2 = np.nan

# EDIT HERE
student_area_ratio = np.nan

pd.Series({
    "A_throat_mm2": student_A_throat_m2 * 1e6,
    "A_outlet_mm2": student_A_outlet_m2 * 1e6,
    "A_outlet_over_A_throat": student_area_ratio,
})

In [ ]:
compare_with_reference(
    "Total throat area",
    student_A_throat_m2,
    reference_design.geometry["A_throat_m2"],
    unit="m²",
)
compare_with_reference(
    "Total nozzle outlet area",
    student_A_outlet_m2,
    reference_design.geometry["A_outlet_m2"],
    unit="m²",
)

## Why does blade height use the axial velocity?

At stator exit,

$$
c_{1a}=c_1\sin\alpha_1,
\qquad
c_{1u}=c_1\cos\alpha_1.
$$

The throat/outlet slot areas above are defined **normal to the local nozzle velocity**, so continuity uses the magnitude $c$.

The annulus area used to size blade height is an **axial plane**, therefore its normal velocity is $c_{1a}$:

$$
\dot m
=
\rho c_{1a}\, e\,\pi D_{\mathrm{mid}}h.
$$

> **STOP — joint discussion:** same continuity equation, different orientation of the control surface.

In [ ]:
pd.Series({
    "c1a_m_s": reference_design.velocities["c1a_m_s"],
    "c1u_m_s": reference_design.velocities["c1u_m_s"],
    "nozzle_height_mm": reference_design.geometry["h_nozzle_m"] * 1e3,
    "rotor_height_mm": reference_design.geometry["h_rotor_m"] * 1e3,
    "nozzle_count": reference_design.geometry["no_nozzles"],
    "rotor_blade_count": reference_design.geometry["no_rotor_blades"],
})

---
# Block C — Rotor velocity triangle, metal angle, and incidence

At rotor inlet,

$$
w_{1a}=c_{1a},
\qquad
w_{1u}=c_{1u}-U,
$$

$$
w_1=\sqrt{w_{1a}^2+w_{1u}^2}.
$$

The incoming flow angle is

$$
\beta_{1,\mathrm{flow}}
=
\tan^{-1}\left(\frac{w_{1a}}{w_{1u}}\right).
$$

The blade geometry does not change in off-design operation. The inlet metal angle is fixed, while the flow angle moves with the velocity triangle. We define incidence as

$$
i
=
\beta_{1,\mathrm{flow}}-\beta_{1,\mathrm{metal}}.
$$

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>EQUATION → PYTHON 4A</b><br>
Reconstruct the rotor inlet triangle. The first line is already completed as an example.
</div>

In [ ]:
c1a_m_s = reference_design.velocities["c1a_m_s"]
c1u_m_s = reference_design.velocities["c1u_m_s"]
U_m_s = reference_design.velocities["U_m_s"]

student_w1a_m_s = c1a_m_s

# EDIT HERE
student_w1u_m_s = np.nan

# EDIT HERE
student_w1_m_s = np.nan

# EDIT HERE
student_beta1_flow_deg = np.nan

# EDIT HERE
student_incidence_deg = np.nan

pd.Series({
    "w1_m_s": student_w1_m_s,
    "beta1_flow_deg": student_beta1_flow_deg,
    "beta1_metal_deg": inputs.beta_rotor_in_deg,
    "incidence_deg": student_incidence_deg,
})

In [ ]:
compare_with_reference(
    "Relative inlet speed",
    student_w1_m_s,
    reference_design.velocities["w1_m_s"],
    unit="m/s",
)
compare_with_reference(
    "Rotor inlet flow angle",
    student_beta1_flow_deg,
    reference_design.velocities["beta1_flow_deg"],
    unit="deg",
)
compare_with_reference(
    "Incidence",
    student_incidence_deg,
    reference_design.velocities["incidence_deg"],
    unit="deg",
)

## Incidence is an off-design loss — not a new blade angle

The empirical rotor loss correlation is a **design-point correlation** and assumes that flow and metal angles are matched.

We therefore keep the geometric blade turning fixed and add incidence as a separate low-order loss correction:

$$
\phi_{r,\mathrm{eff}}
=
\sqrt{
\phi_{r,0}^2
-
K_i\sin^2 i
}.
$$

In this teaching model $K_i=1$.

At rotor exit we currently neglect deviation and assume that the relative outlet flow follows the metal angle. This is an explicit model simplification.

In [ ]:
pd.Series({
    "beta1_flow_deg": reference_design.velocities["beta1_flow_deg"],
    "beta1_metal_deg": reference_design.velocities["beta1_metal_deg"],
    "incidence_deg": reference_design.velocities["incidence_deg"],
    "phi_rotor_zero_incidence": reference_design.performance["phi_rotor_zero_incidence"],
    "phi_rotor_effective": reference_design.performance["phi_rotor"],
    "beta2_metal_deg": inputs.beta_rotor_out_deg,
    "c2u_m_s": reference_design.velocities["c2u_m_s"],
})

In [ ]:
fig, _ = plot_velocity_triangles(reference_design.velocities)
plt.show()

> **STOP — joint discussion**
>
> Why can a fixed blade have a changing flow angle?  
> What happens to incidence if rpm changes but the metal angle does not?  
> Why do we keep $\beta_{1,\mathrm{metal}}$ and $\beta_{2,\mathrm{metal}}$ fixed in off-design operation?

---
# Block D — Euler work, losses, and turbine efficiency

Euler's turbine equation gives the aerodynamic power

$$
P_{\mathrm{aero}}
=
\dot m U(c_{1u}-c_{2u}).
$$

The complete teaching model subsequently subtracts disk-friction and partial-admission losses.

<div style="padding:0.8rem 1rem;background:#fff3e0;border-left:6px solid #ef6c00;border-radius:5px">
<b>EQUATION → PYTHON 4B</b><br>
One final line: translate Euler's turbine equation.
</div>

In [ ]:
c2u_m_s = reference_design.velocities["c2u_m_s"]

# EDIT HERE
student_P_aero_W = np.nan

pd.Series({
    "P_aero_kW": student_P_aero_W / 1e3,
})

In [ ]:
compare_with_reference(
    "Aerodynamic power",
    student_P_aero_W,
    reference_design.performance["P_aero_W"],
    unit="W",
)

## Where does the aerodynamic power go?

The reference implementation now adds the empirical loss models and returns the complete stage result.

Residual rotor-exit swirl $c_{2u}\neq0$ means that tangential kinetic energy remains in the flow.  
However, forcing $c_{2u}=0$ is not automatically the optimum if the required blade turning causes larger rotor losses.

In [ ]:
plot_loss_breakdown(reference_design)
plt.show()

reference_design.summary()

### Optional 60-second geometry check

From the mean diameter and blade height,

$$
D_{\mathrm{hub}}=D_{\mathrm{mid}}-h,
\qquad
D_{\mathrm{tip}}=D_{\mathrm{mid}}+h.
$$

You can calculate these by hand and compare them with the summary table above. No notebook editing is required.

---
# Design playground — experiment instead of coding

Work in pairs. Choose **one** control variable in the explorer:

- rotational speed $n$,
- mean diameter $D_{\mathrm{mid}}$,
- stator angle $\alpha_1$,
- rotor inlet metal angle $\beta_1$,
- rotor outlet metal angle $\beta_2$,
- pressure ratio.

Before moving the slider:

1. predict the direction of change in at least two outputs;
2. move only one slider;
3. explain the result using the equations from Blocks A–D.

Good outputs to watch are $\eta$, $P_{\mathrm{mech}}$, $U/c_{\mathrm{is}}$, incidence, $\phi_r$, and $c_{2u}$.

In [ ]:
design_explorer(inputs)

---
# Block E — Reduced off-design characteristic for TESPy

<div style="padding:0.8rem 1rem;background:#eef4ff;border-left:6px solid #1565c0;border-radius:5px">
<b>RUN-ONLY SECTION.</b><br>
There is no student coding task here. This is the final connection between the turbine design workshop and Francesco's TESPy workshop.
</div>

The geometry is now **frozen**.

For each pressure ratio,

$$
\Pi=\frac{p_{\mathrm{in}}}{p_{\mathrm{out}}},
$$

the model tests a range of speeds and keeps the speed with the highest turbine efficiency:

$$
n_{\mathrm{opt}}=n_{\mathrm{opt}}(\Pi),
$$

$$
\eta_{\mathrm{is,ts,opt}}
=
\eta_{\mathrm{is,ts}}(\Pi,n_{\mathrm{opt}}).
$$

TESPy therefore receives a one-dimensional efficiency characteristic $\eta_{\mathrm{is,ts}}(\Pi)$.  
The optimum rpm is informative post-processing rather than an independent cycle input.

## Mass flow is imposed by the fixed sonic throat

For choked operation,

$$
\dot m
=
C_d A^*\rho^*a^*.
$$

The sonic state $*$ is calculated from the **current turbine inlet state** along the inlet isentrope.

This is important:

- efficiency is interpolated from the reduced pressure-ratio characteristic;
- mass flow should ultimately be recalculated from the fixed throat and the current TESPy inlet state;
- the tabulated $\dot m$ below is only the reference sliding-pressure trajectory used to generate this map.

In [ ]:
geometry = freeze_geometry(reference_design, backend=backend)

p_critical_Pa = backend.critical_pressure(geometry.fluid)

PR_min = max(1.6, 0.45 * geometry.pressure_ratio_design)
PR_max = min(
    1.60 * geometry.pressure_ratio_design,
    0.92 * p_critical_Pa / geometry.p_out_design_Pa,
)

PR_values = np.linspace(PR_min, PR_max, 32)

n_search_values = np.linspace(
    0.35 * geometry.n_design_rpm,
    2.00 * geometry.n_design_rpm,
    72,
)

print(f"Design pressure ratio: {geometry.pressure_ratio_design:.3f}")
print(f"Pressure-ratio sweep: {PR_min:.3f} ... {PR_max:.3f}")
print(
    f"Internal speed search: "
    f"{n_search_values[0]:.0f} ... {n_search_values[-1]:.0f} rpm"
)

In [ ]:
records = []
skipped = []

for PR in PR_values:
    p_in_Pa = PR * geometry.p_out_design_Pa

    T_in_K = inlet_temperature_for_pressure(
        geometry,
        p_in_Pa,
        backend=backend,
    )

    stator = evaluate_stator_offdesign(
        geometry,
        p_in_Pa,
        T_in_K,
        geometry.p_out_design_Pa,
        backend=backend,
    )

    if not stator["valid_stator"] or not stator["choked"]:
        skipped.append((PR, "stator not choked/valid"))
        continue

    candidates = []

    for n_rpm in n_search_values:
        point = evaluate_rotor_offdesign(
            geometry,
            stator,
            n_rpm,
            backend=backend,
        )
        if point["valid"]:
            candidates.append(point)

    if not candidates:
        skipped.append((PR, "no valid rotor speed"))
        continue

    optimum = max(candidates, key=lambda point: point["eta_turb"])

    records.append({
        "pressure_ratio": float(PR),
        "p_in_Pa": float(p_in_Pa),
        "T_in_K": float(T_in_K),
        "p_out_Pa": float(geometry.p_out_design_Pa),
        "m_dot_kg_s": float(stator["m_dot_kg_s"]),
        "eta_is_ts": float(optimum["eta_turb"]),
        "P_mech_W": float(optimum["P_mech_W"]),
        "n_opt_rpm": float(optimum["n_rpm"]),
        "U_over_cis_opt": float(optimum["U_over_cis"]),
        "Ma_nozzle_out": float(stator["Ma_nozzle_out"]),
        "area_mismatch_rel": float(stator["area_mismatch_rel"]),
        "choked": bool(stator["choked"]),
    })

reduced_map = pd.DataFrame.from_records(records)

if reduced_map.empty:
    raise RuntimeError("No valid choked off-design points were found.")

print(f"Retained {len(reduced_map)} of {len(PR_values)} pressure-ratio points.")
if skipped:
    print(f"Skipped {len(skipped)} points outside the valid model region.")

reduced_map.head(8)

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(reduced_map["pressure_ratio"], reduced_map["eta_is_ts"], marker="o")
ax.axvline(
    geometry.pressure_ratio_design,
    linestyle="--",
    linewidth=1.0,
    label="Design pressure ratio",
)
ax.set_xlabel("Pressure ratio [-]")
ax.set_ylabel("Total-to-static efficiency [-]")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(reduced_map["pressure_ratio"], reduced_map["n_opt_rpm"], marker="o")
ax.axvline(
    geometry.pressure_ratio_design,
    linestyle="--",
    linewidth=1.0,
    label="Design pressure ratio",
)
ax.set_xlabel("Pressure ratio [-]")
ax.set_ylabel("Optimum rotational speed [rpm]")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(reduced_map["pressure_ratio"], reduced_map["m_dot_kg_s"], marker="o")
ax.axvline(
    geometry.pressure_ratio_design,
    linestyle="--",
    linewidth=1.0,
    label="Design pressure ratio",
)
ax.set_xlabel("Pressure ratio [-]")
ax.set_ylabel("Reference choked mass flow [kg/s]")
ax.legend()
fig.tight_layout()
plt.show()

## Export the handoff file

The JSON contains the fixed throat area, the reduced efficiency characteristic, optimum rpm, and the reference sliding-pressure trajectory.

For the TESPy model, the physical mass-flow residual is conceptually

$$
R_{\dot m}
=
\dot m_{\mathrm{TESPy}}
-
C_dA^*\rho^*a^*
=
0.
$$

In [ ]:
export_data = {
    "schema": "kcorc_reduced_turbine_characteristic_v1",
    "fluid": geometry.fluid,

    "design_point": {
        "p_in_Pa": float(geometry.p_in_design_Pa),
        "T_in_K": float(geometry.T_in_design_K),
        "p_out_Pa": float(geometry.p_out_design_Pa),
        "pressure_ratio": float(geometry.pressure_ratio_design),
        "m_dot_kg_s": float(geometry.m_dot_design_kg_s),
        "n_rpm": float(geometry.n_design_rpm),
        "eta_is_ts": float(geometry.eta_design),
        "P_mech_W": float(geometry.P_mech_design_W),
    },

    "fixed_geometry": {
        "D_mid_m": float(geometry.D_mid_m),
        "A_throat_m2": float(geometry.A_throat_m2),
        "A_outlet_m2": float(geometry.A_outlet_m2),
        "h_rotor_m": float(geometry.h_rotor_m),
        "partial_admission": float(geometry.partial_admission_eff),
    },

    "mass_flow_model": {
        "type": "fixed_choked_throat",
        "equation": "m_dot = Cd * A_throat * rho_star * a_star",
        "discharge_coefficient": 1.0,
        "note": (
            "The sonic state is evaluated from the current turbine inlet state. "
            "The tabulated mass-flow values below are only the reference "
            "sliding-pressure trajectory."
        ),
    },

    "map_assumptions": {
        "geometry": "fixed",
        "p_out_Pa": float(geometry.p_out_design_Pa),
        "inlet_path": (
            "sliding pressure with design superheat preserved where "
            "a saturation state is available"
        ),
        "speed_control": (
            "ideal variable-speed operation; maximum eta_is_ts selected "
            "at each pressure ratio"
        ),
        "validity": "only choked stator points passing the reduced-model checks",
    },

    "characteristic": {
        "pressure_ratio": reduced_map["pressure_ratio"].tolist(),
        "eta_is_ts": reduced_map["eta_is_ts"].tolist(),
        "n_opt_rpm": reduced_map["n_opt_rpm"].tolist(),
        "m_dot_kg_s": reduced_map["m_dot_kg_s"].tolist(),
        "P_mech_W": reduced_map["P_mech_W"].tolist(),
        "p_in_Pa": reduced_map["p_in_Pa"].tolist(),
        "T_in_K": reduced_map["T_in_K"].tolist(),
        "U_over_cis_opt": reduced_map["U_over_cis_opt"].tolist(),
    },
}

map_file = OUTPUT_DIR / "turbine_reduced_characteristic.json"

with map_file.open("w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)

print(f"Saved TESPy handoff to: {map_file.resolve()}")

---
# Final discussion

You should now be able to explain the model without explaining the Python.

1. Why does a high ORC pressure ratio naturally lead to a supersonic stator?
2. What fixes the mass flow once the nozzle throat is choked?
3. Why is $U/c_{\mathrm{is}}$ a useful turbine-design parameter?
4. Why are metal angles fixed while flow angles change in off-design operation?
5. What does rotor incidence do to the effective rotor velocity coefficient?
6. Why is $c_{2u}=0$ desirable but not necessarily the global efficiency optimum?
7. Which quantities should the turbine model pass to a cycle model such as TESPy?

### Take-home message

The objective was not to learn Python syntax.  
The objective was to connect

**cycle boundary conditions → real-fluid expansion → nozzle → velocity triangles → losses → geometry → reduced off-design characteristic**.